# FrogNano-4B — Leaf coding agent on a free T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/main/colab/FrogNano-4B-coding-agent.ipynb)

[`microsoft/FrogNano-4B-2609`](https://huggingface.co/microsoft/FrogNano-4B-2609) is Microsoft's ~4.66B repository-level coding agent, post-trained with reinforcement learning on about 1,500 synthetic SWE tasks from [`Qwen/Qwen3.5-4B`](https://huggingface.co/Qwen/Qwen3.5-4B). It is meant to be driven through the lightweight **Leaf** harness: five typed tools (`Read`, `Write`, `Edit`, `Glob`, `Bash`). A response that contains tool calls continues the episode; a response without one is the final answer.

This notebook is a **mini Leaf loop**, not a unified-diff agent. It embeds a tiny failing pytest invoice repo, lets the model inspect and edit through those five tools, and reruns the tests. Official evals use 150 steps and ~131K context; the demo caps the loop at **8 Leaf turns** (about 2–3 inspect / edit / test cycles).

Native BF16 weights are about **9.3 GB** before KV cache and runtime state. That is tight on a 16 GB T4, so the walkthrough loads the bartowski **Q4_K_M GGUF** (~2.80 GB) with `llama-cpp-python`.

**Licence:** the Hub tag is **MIT**. The model-card table also says **Apache License 2.0** because FrogNano is derived from Qwen3.5-4B; keep the upstream notices.

**Runtime:** *Runtime > Change runtime type > T4 GPU*. The free T4 has ~15 GB of VRAM, **no native bfloat16**, and **no FlashAttention-2**. Do not load BF16 or turn FlashAttention on. This notebook keeps `flash_attn=False` (llama.cpp's SDPA / eager path) and allocates **16,384** tokens of context. The evaluated 131K window is not allocated.

**Links:** [model card](https://huggingface.co/microsoft/FrogNano-4B-2609) · [GGUF](https://huggingface.co/bartowski/FrogNano-4B-2609-GGUF) · [Leaf harness](https://github.com/microsoft/FrogNano) · [technical report](https://aka.ms/frognano-tech-report)


## 1. Install the CUDA `llama-cpp-python` wheel

Remove any CPU build and install the pinned CUDA 12.1 wheel from the project's GitHub releases. A direct wheel URL stops pip from grabbing the CPU package on PyPI. CUDA 12.1 binaries work with the newer driver on a Colab T4. `pytest` is the test runner the Leaf `Bash` tool will call.

FrogNano is a Qwen3.5-4B checkpoint. **0.3.36** (1 October 2026) is the current CUDA 12.1 wheel and is after llama.cpp gained that architecture. The GGUF was built with llama.cpp `b11279`.


In [ ]:
%pip uninstall -qqq -y llama-cpp-python
%pip install -qqq --no-cache-dir https://github.com/abetlen/llama-cpp-python/releases/download/v0.3.36-cu121/llama_cpp_python-0.3.36-py3-none-manylinux_2_35_x86_64.whl
%pip install -qqq pytest

# Direct wheel URL, same pattern as colab/LFM-2_5-2_6b-llamacpp-python.ipynb.
# 0.3.36 is after llama.cpp gained the Qwen3.5 architecture used by FrogNano.


> **Restart the runtime now:** *Runtime > Restart session*, then continue from the next cell.
> Native libraries already imported cannot be replaced in the same process. If you skip this after the install, the next cell stops with a version check.


## 2. Check the GPU and print T4 caveats


In [ ]:
import llama_cpp
import torch
from packaging.version import Version

assert Version(llama_cpp.__version__) >= Version("0.3.36"), (
    f"llama-cpp-python {llama_cpp.__version__} is loaded; FrogNano GGUF needs 0.3.36+. "
    "Run the install cell, then Runtime > Restart session."
)

if not torch.cuda.is_available():
    raise SystemExit("Select Runtime > Change runtime type > T4 GPU, then rerun from the top.")

!nvidia-smi

major, minor = torch.cuda.get_device_capability()
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
name = torch.cuda.get_device_name()

print()
print("=== T4-first caveats ===")
print(f"GPU: {name}  sm_{major}{minor}  VRAM: {vram_gb:.1f} GB")
print(f"llama-cpp-python {llama_cpp.__version__}")
print("- No native bfloat16 on T4 (sm_75). This notebook loads Q4_K_M GGUF, not BF16.")
print("- Native BF16 / fp16 weights are ~9.3 GB and leave little room for KV + runtime on 15 GB.")
print("- No FlashAttention-2 on sm_75. flash_attn=False (sdpa / eager equivalent).")
print("- bartowski Q4_K_M GGUF is ~2.80 GB; 16K context still fits comfortably on 15 GB.")
print("- The evaluated 131K window is not allocated. n_ctx=16384.")
print("- Official Leaf evals use 150 steps; this demo caps the loop at 8 turns.")
print("- Thinking traces can spend tokens; generation is capped per turn below.")
if major < 8:
    print("- FlashAttention stays off on this GPU.")


## 3. Load the Q4_K_M GGUF

`Llama.from_pretrained` downloads `FrogNano-4B-2609-Q4_K_M.gguf` (~2.80 GB) from [`bartowski/FrogNano-4B-2609-GGUF`](https://huggingface.co/bartowski/FrogNano-4B-2609-GGUF). `n_gpu_layers=-1` offloads every layer to the T4. `n_ctx=16384` leaves room for the toy repo, a short think block, and a few tool observations without touching the 131K evaluation window. `flash_attn=False` is required on sm_75.

Sampler defaults match the official FrogNano eval presets: temperature 0.6, top_p 0.95, top_k 20, repetition penalty 1.0.


In [ ]:
import os

os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

from llama_cpp import Llama

REPO_ID = "bartowski/FrogNano-4B-2609-GGUF"
FILENAME = "FrogNano-4B-2609-Q4_K_M.gguf"
N_CTX = 16384

# Official Leaf / FrogNano eval sampler (thinking stays on via the GGUF chat template).
SAMPLER = dict(temperature=0.6, top_p=0.95, top_k=20, repeat_penalty=1.0)

llm = Llama.from_pretrained(
    repo_id=REPO_ID,
    filename=FILENAME,
    n_gpu_layers=-1,
    n_ctx=N_CTX,
    flash_attn=False,
    verbose=True,
)

print()
print("loaded", FILENAME, "| n_ctx", N_CTX, "| flash_attn False")
print("sampler", SAMPLER)
print()
print("GPU status after load (llama.cpp VRAM is outside the PyTorch allocator):")
!nvidia-smi


## 4. In-process Leaf tools

Official Leaf (see [`microsoft/FrogNano`](https://github.com/microsoft/FrogNano)) exposes five OpenAI-schema tools and a short system prompt. The Kubernetes sandbox is not available in Colab, so this cell keeps the same names, parameters, and termination rule and runs them against a local repo root.

- `Read` — numbered lines, optional `offset` / `limit`
- `Write` — full-file write, creates parents
- `Edit` — exact-string replace (`replace_all` if the old string matches more than once)
- `Glob` — file search from the repo root
- `Bash` — `/bin/bash -lc` with cwd set to the repo

A response with tool calls continues; a response without one ends the episode.


In [ ]:
import glob as globlib
import json
import os
import re
import subprocess
from pathlib import Path
from typing import Any

# Official Leaf system prompt from frognano/harness/leaf/tools.py
SYSTEM_PROMPT = (
    "You are a software-engineering agent working in a checked-out repository.\n"
    "Use the available tools to inspect files, make focused edits, and run commands.\n"
    "Solve the task, validate when practical, then give a brief final answer."
)

# Official Leaf OpenAI tool schemas (Read / Write / Edit / Glob / Bash).
OPENAI_TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "Read",
            "description": "Read a text file. Returns numbered lines.",
            "parameters": {
                "type": "object",
                "properties": {
                    "file_path": {"type": "string", "description": "Path to the file to read."},
                    "offset": {"type": "integer", "description": "Optional 1-based starting line."},
                    "limit": {"type": "integer", "description": "Optional maximum number of lines."},
                },
                "required": ["file_path"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "Write",
            "description": "Write a complete file, creating parent directories when needed.",
            "parameters": {
                "type": "object",
                "properties": {
                    "file_path": {"type": "string", "description": "Path to write."},
                    "content": {"type": "string", "description": "Full file contents."},
                },
                "required": ["file_path", "content"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "Edit",
            "description": "Replace an exact string in a text file.",
            "parameters": {
                "type": "object",
                "properties": {
                    "file_path": {"type": "string", "description": "Path to edit."},
                    "old_string": {"type": "string", "description": "Exact text to replace."},
                    "new_string": {"type": "string", "description": "Replacement text."},
                    "replace_all": {
                        "type": "boolean",
                        "description": "Replace every occurrence instead of requiring one match.",
                    },
                },
                "required": ["file_path", "old_string", "new_string"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "Glob",
            "description": "Find files by glob pattern.",
            "parameters": {
                "type": "object",
                "properties": {
                    "pattern": {"type": "string", "description": "Glob pattern, e.g. **/*.py."},
                    "path": {"type": "string", "description": "Optional directory to search from."},
                },
                "required": ["pattern"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "Bash",
            "description": "Run a shell command in the repository workdir.",
            "parameters": {
                "type": "object",
                "properties": {
                    "command": {"type": "string", "description": "Command to run."},
                    "description": {"type": "string", "description": "Short description of the command."},
                    "timeout": {"type": "number", "description": "Optional timeout in seconds."},
                },
                "required": ["command"],
            },
        },
    },
]

REPO = Path("/content/toy-invoice") if Path("/content").exists() else Path("toy-invoice")
REPO = REPO.resolve()
TOOL_OUTPUT_CHARS = 20_000
BASH_OUTPUT_CHARS = 30_000
GLOB_LIMIT = 200

_TOOL_CALL_RE = re.compile(
    r"<tool_call>\s*<function=([^>]+)>\s*(.*?)</function>\s*</tool_call>",
    re.S,
)
_PARAM_RE = re.compile(r"<parameter=([^>]+)>\s*(.*?)\s*</parameter>", re.S)


def _truthy(value: Any) -> bool:
    if isinstance(value, bool):
        return value
    return str(value or "").strip().lower() in {"1", "true", "yes", "on"}


def _optional_int(value: Any, default: int) -> int:
    if value in (None, ""):
        return default
    return int(value)


def _truncate(value: str, limit: int) -> str:
    if limit <= 0 or len(value) <= limit:
        return value
    return f"{value[:limit]}\n[truncated {len(value) - limit} chars]"


def resolve_repo_path(value: Any) -> Path:
    path = Path(str(value)).expanduser()
    if not path.is_absolute():
        path = REPO / path
    path = path.resolve()
    if path != REPO and REPO not in path.parents:
        raise ValueError(f"path escapes the repo: {value}")
    return path


def display_path(path: Path) -> str:
    try:
        return path.resolve().relative_to(REPO).as_posix()
    except ValueError:
        return str(path)


def run_leaf_tool(name: str, arguments: dict[str, Any]) -> str:
    try:
        if name == "Read":
            path = resolve_repo_path(arguments.get("file_path") or arguments.get("path"))
            lines = path.read_text(encoding="utf-8", errors="replace").splitlines()
            offset = max(1, _optional_int(arguments.get("offset"), 1))
            limit = _optional_int(arguments.get("limit"), 0)
            end = len(lines) if limit <= 0 else min(len(lines), offset - 1 + limit)
            selected = "\n".join(
                f"{number}. {line}"
                for number, line in enumerate(lines[offset - 1 : end], start=offset)
            )
            return _truncate(f"{display_path(path)}\n{selected}", TOOL_OUTPUT_CHARS)
        if name == "Write":
            path = resolve_repo_path(arguments.get("file_path") or arguments.get("path"))
            content = str(arguments.get("content", ""))
            path.parent.mkdir(parents=True, exist_ok=True)
            path.write_text(content, encoding="utf-8")
            return f"Wrote {display_path(path)} ({len(content)} bytes)."
        if name == "Edit":
            path = resolve_repo_path(arguments.get("file_path") or arguments.get("path"))
            old = str(arguments.get("old_string") or "")
            new = str(arguments.get("new_string") or "")
            if not old:
                raise ValueError("old_string must be non-empty")
            text = path.read_text(encoding="utf-8", errors="replace")
            count = text.count(old)
            replace_all = _truthy(arguments.get("replace_all"))
            if count == 0:
                raise ValueError("old_string was not found")
            if count != 1 and not replace_all:
                raise ValueError(
                    f"old_string matched {count} times; set replace_all=true or use a "
                    "more specific old_string"
                )
            updated = text.replace(old, new) if replace_all else text.replace(old, new, 1)
            path.write_text(updated, encoding="utf-8")
            changed = count if replace_all else 1
            suffix = "s" if changed != 1 else ""
            return f"Edited {display_path(path)} ({changed} replacement{suffix})."
        if name == "Glob":
            pattern = str(arguments.get("pattern") or "")
            if not pattern:
                raise ValueError("pattern is required")
            root = resolve_repo_path(arguments.get("path") or ".")
            matches = sorted(
                display_path(Path(match))
                for match in globlib.glob(str(root / pattern), recursive=True)
                if Path(match).is_file()
            )
            shown = matches[:GLOB_LIMIT]
            suffix = "" if len(matches) <= GLOB_LIMIT else f"\n... {len(matches) - GLOB_LIMIT} more files"
            return "\n".join(shown) + suffix if shown else "No files matched."
        if name == "Bash":
            command = str(arguments.get("command") or "")
            if not command:
                raise ValueError("command is required")
            raw_timeout = arguments.get("timeout")
            timeout = 30.0 if raw_timeout is None else float(raw_timeout)
            if timeout > 1000:
                timeout /= 1000.0
            timeout = max(0.1, timeout)
            completed = subprocess.run(
                ["/bin/bash", "-lc", command],
                cwd=REPO,
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
                universal_newlines=True,
                timeout=timeout,
                check=False,
            )
            return _truncate(
                f"Exit code: {completed.returncode}\n"
                f"STDOUT:\n{completed.stdout or ''}\n"
                f"STDERR:\n{completed.stderr or ''}",
                BASH_OUTPUT_CHARS,
            )
        return f"Unknown tool: {name}"
    except subprocess.TimeoutExpired as exc:
        return _truncate(
            f"Timed out after {exc.timeout:.1f}s\n"
            f"STDOUT:\n{exc.stdout or ''}\nSTDERR:\n{exc.stderr or ''}",
            BASH_OUTPUT_CHARS,
        )
    except Exception as exc:
        return f"{type(exc).__name__}: {exc}"


def parse_xml_tool_calls(text: str) -> list[dict[str, Any]]:
    calls = []
    for index, match in enumerate(_TOOL_CALL_RE.finditer(text or ""), start=1):
        name = match.group(1).strip()
        arguments: dict[str, Any] = {}
        for param in _PARAM_RE.finditer(match.group(2)):
            key = param.group(1).strip()
            value = param.group(2)
            if key in {"offset", "limit"}:
                try:
                    arguments[key] = int(value.strip())
                    continue
                except ValueError:
                    pass
            if key in {"replace_all"}:
                arguments[key] = _truthy(value)
                continue
            if key == "timeout":
                try:
                    arguments[key] = float(value.strip())
                    continue
                except ValueError:
                    pass
            arguments[key] = value
        calls.append({"id": f"call_xml_{index}", "name": name, "arguments": arguments})
    return calls


def parse_native_tool_calls(message: dict[str, Any]) -> list[dict[str, Any]]:
    calls = []
    for index, raw in enumerate(message.get("tool_calls") or [], start=1):
        function = raw.get("function") or raw
        name = function.get("name") or raw.get("name")
        arguments = function.get("arguments") or raw.get("arguments") or {}
        if isinstance(arguments, str):
            try:
                arguments = json.loads(arguments) if arguments.strip() else {}
            except json.JSONDecodeError:
                arguments = {}
        if not isinstance(arguments, dict):
            arguments = {}
        calls.append(
            {
                "id": raw.get("id") or f"call_{index}",
                "name": name,
                "arguments": arguments,
            }
        )
    return [call for call in calls if call.get("name")]


def split_think(raw: str) -> tuple[str, str]:
    if "</think>" in raw:
        think, answer = raw.split("</think>", 1)
        think = think.split("<think>", 1)[-1].strip()
        return think, answer.strip()
    return "", (raw or "").strip()


print("Leaf tools:", ", ".join(t["function"]["name"] for t in OPENAI_TOOLS))
print("repo root will be", REPO)


## 5. Embed a tiny failing invoice repo

A two-file invoice helper with three intentional bugs. The prompt does **not** dump the source: FrogNano is supposed to `Glob` / `Read` its way in. Everything lives under `/content/toy-invoice` so the notebook does not depend on a network clone.


In [ ]:
import subprocess
import sys
import textwrap

REPO.mkdir(parents=True, exist_ok=True)
(REPO / "tests").mkdir(parents=True, exist_ok=True)

(REPO / "README.md").write_text(
    textwrap.dedent(
        """
        # toy-invoice

        Tiny invoice helpers. `pytest` in this directory is the contract.

        - `line_total(qty, unit_price)` must return quantity times unit price.
        - `apply_discount(amount, percent)` must subtract `percent` percent.
        - `apply_tax(amount, rate)` must add a fractional tax rate (`0.08` means 8%).
        - `invoice_total` composes those three helpers. Do not change the tests.
        """
    ).lstrip()
)

(REPO / "invoice.py").write_text(
    textwrap.dedent(
        """
        # Invoice helpers. Several functions have intentional bugs.


        def line_total(qty, unit_price):
            # Quantity times unit price.
            return qty + unit_price


        def apply_discount(amount, percent):
            # Subtract `percent` percent from amount.
            return amount * percent / 100.0


        def apply_tax(amount, rate):
            # Add a fractional tax rate (0.08 means 8%).
            return amount * rate
        """
    ).lstrip()
)

(REPO / "totals.py").write_text(
    textwrap.dedent(
        """
        from invoice import apply_discount, apply_tax, line_total


        def invoice_total(items, discount_percent, tax_rate):
            subtotal = sum(line_total(qty, price) for qty, price in items)
            return apply_tax(apply_discount(subtotal, discount_percent), tax_rate)
        """
    ).lstrip()
)

(REPO / "tests" / "test_invoice.py").write_text(
    textwrap.dedent(
        """
        import sys
        from pathlib import Path

        sys.path.insert(0, str(Path(__file__).resolve().parents[1]))

        from invoice import apply_discount, apply_tax, line_total
        from totals import invoice_total


        def test_line_total():
            assert line_total(3, 10) == 30
            assert line_total(0, 5) == 0


        def test_apply_discount():
            assert apply_discount(100, 10) == 90
            assert apply_discount(50, 0) == 50


        def test_apply_tax():
            assert apply_tax(100, 0.08) == 108
            assert apply_tax(25, 0) == 25


        def test_invoice_total():
            assert invoice_total([(2, 10), (1, 5)], 10, 0.0) == 22.5
        """
    ).lstrip()
)

print("repo tree:")
for path in sorted(p for p in REPO.rglob("*") if p.is_file()):
    print(" ", path.relative_to(REPO), f"({path.stat().st_size} bytes)")

print()
print("=== pytest before the agent ===")
before = subprocess.run(
    [sys.executable, "-m", "pytest", "-q"],
    cwd=REPO,
    text=True,
    capture_output=True,
)
print(before.stdout)
print(before.stderr)
print("exit", before.returncode)
assert before.returncode != 0, "the toy repo should fail before the agent runs"


## 6. Mini Leaf loop (8 turns, about 2–3 inspect / edit / test cycles)

The user message is the issue only — no source dump. FrogNano should `Glob` the tree, `Read` the failing files, `Edit` (or `Write`) the bugs, and `Bash` `pytest`. The chat template can emit OpenAI `tool_calls` or Qwen `<tool_call>` XML; both are executed.

`max_tokens=2048` bounds a think block plus one or more tool calls. Official Leaf allows 8,192 tokens per turn and 150 steps; this demo stops after 8 assistant turns or when the model answers without tools. pytest is also run at the end so a silent patch is still graded.


In [ ]:
import json
import subprocess
import sys

MAX_STEPS = 8
MAX_TOKENS = 2048


def leaf_turn(messages, max_tokens=MAX_TOKENS):
    """One FrogNano turn. Returns (message_dict, think, answer, tool_calls, raw)."""
    response = llm.create_chat_completion(
        messages=messages,
        tools=OPENAI_TOOLS,
        tool_choice="auto",
        max_tokens=max_tokens,
        stream=False,
        **SAMPLER,
    )
    message = response["choices"][0]["message"]
    raw = message.get("content") or ""
    think, answer = split_think(raw)
    tool_calls = parse_native_tool_calls(message)
    if not tool_calls:
        tool_calls = parse_xml_tool_calls(raw)
    return message, think, answer, tool_calls, raw


instruction = (
    f"The invoice helpers in this repository fail pytest.\n"
    f"Repo root: {REPO}\n\n"
    "Requirements:\n"
    "- line_total(qty, unit_price) returns qty * unit_price\n"
    "- apply_discount(amount, percent) subtracts `percent` percent of amount\n"
    "- apply_tax(amount, rate) adds a fractional tax rate (0.08 means 8%)\n"
    "- invoice_total composes those helpers; do not change the tests\n\n"
    "Use the tools to inspect the repo, edit the code, and run `python -m pytest -q` "
    "until it passes. Then give a brief final answer. Do not write a unified diff; call the tools."
)

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": instruction},
]

passed = False
exit_reason = "max_turns"
for step in range(1, MAX_STEPS + 1):
    print(f"\n========== Leaf turn {step}/{MAX_STEPS} ==========\n")
    message, think, answer, tool_calls, raw = leaf_turn(messages)
    assistant = {"role": "assistant", "content": raw}
    if message.get("tool_calls"):
        assistant["tool_calls"] = message["tool_calls"]
    elif tool_calls:
        # XML-only path: keep the raw <tool_call> text and do not also
        # re-emit structured tool_calls (the chat template would duplicate them).
        pass
    messages.append(assistant)

    if think:
        preview = think if len(think) <= 600 else think[:600] + "\n..."
        print(f"--- think ({len(think.split())} words) ---\n{preview}\n")
    if answer:
        print("--- assistant ---\n" + (answer if len(answer) <= 1200 else answer[:1200] + "\n..."))
    elif not tool_calls:
        print("--- assistant ---\n(empty)")

    if not tool_calls:
        exit_reason = "agent"
        print("\nno tool calls; Leaf episode ends.")
        break

    for call in tool_calls:
        print(f"\n>>> {call['name']} {json.dumps(call['arguments'], ensure_ascii=False)}")
        observation = run_leaf_tool(call["name"], call["arguments"])
        print(observation if len(observation) <= 1500 else observation[:1500] + "\n...")
        messages.append(
            {
                "role": "tool",
                "tool_call_id": call["id"],
                "content": observation,
            }
        )

print()
print("=== pytest after the Leaf loop ===")
after = subprocess.run(
    [sys.executable, "-m", "pytest", "-q"],
    cwd=REPO,
    text=True,
    capture_output=True,
)
print(after.stdout)
print(after.stderr)
print("exit", after.returncode)
print("exit_reason", exit_reason)
passed = after.returncode == 0

print()
print("=== invoice.py after the loop ===")
print((REPO / "invoice.py").read_text())
if passed:
    print("agent resolved the suite.")
else:
    print(f"agent did not clear the suite in {MAX_STEPS} Leaf turns.")


## 7. Caveats

- **Licence.** Hub metadata is MIT. The card's summary table lists Apache License 2.0 because FrogNano is derived from Qwen3.5-4B; retain the upstream copyright and attribution notices.
- **Dtype / attention.** The card's documented rollout is SGLang on GPU servers in **BF16**. A free T4 has no native bfloat16 and no FlashAttention-2. This notebook uses Q4_K_M GGUF and `flash_attn=False` (sdpa / eager). Native fp16 (~9.3 GB of weights) can fit a T4 in isolation but leaves little room for KV cache and the tool loop.
- **Leaf harness.** Official Leaf runs inside isolated Kubernetes pods with 150 steps, ~131K context, and up to 8,192 generated tokens per turn. This notebook is an in-process subset: same five tools, same short system prompt, same “no tool call ⇒ stop” rule, 16K context, 8 turns, 2,048 tokens per turn. It is not the eval harness and will not reproduce SWE-bench numbers.
- **Scope.** FrogNano is specialized for English, Python-heavy repos with executable tests. Image / video components are inherited from Qwen3.5-4B and were **not** post-trained. Generated patches can pass tests and still be wrong or insecure; they need human review.
- **Safety.** The card is explicit that coding-focused RL is not a safety alignment pass. Keep the sandbox local, do not point `Bash` at credentials or production systems, and treat every edit as a candidate patch.
